In [2]:
from dotenv import load_dotenv
import os
from typing import Optional, Dict, Any, Tuple
import requests
import time
import pandas as pd

In [3]:
load_dotenv()
API_URL = "https://newsapi.org/v2/everything"
API_KEY = os.getenv('newsapi_apikey') 
PAGE_SIZE = 100

In [4]:
def fetch_news(
    api_url: str,
    params: Dict[str, Any],
    headers: Dict[str, str] = {},
    retries: int = 3,
    backoff_factor: float = 1.0,
    rate_limit_sleep: float = 60.0,
) -> Tuple[Dict[str, Any], Dict[str, str]]:
    """
    Fetch one page of news from API with retry and rate limit handling.

    Args:
        api_url: Full API endpoint URL.
        params: Query parameters for the request.
        headers: Headers for authentication or content-type.
        retries: Number of retry attempts for failed requests.
        backoff_factor: Time to wait between retries (increasing each time).
        rate_limit_sleep: Time to wait (in seconds) if rate limit is hit (HTTP 429).

    Returns:
        Parsed JSON response or None if request failed.
    """
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(api_url, params = params, headers = headers, timeout = 30)
            if response.status_code == 429:
                print("[WARN] Rate limit hit. Sleeping for", rate_limit_sleep, "seconds.")
                time.sleep(rate_limit_sleep)
                continue
            response.raise_for_status()
            return response.json(), response.headers
        except requests.RequestException as e:
            print(f"[ERROR] Attempt {attempt} failed: {e}")
            if attempt < retries:
                sleep_time = backoff_factor * attempt
                print(f"[INFO] Retrying in {sleep_time:.1f} seconds...")
                time.sleep(sleep_time)
            else:
                print("[ERROR] Max retries reached. Giving up.")
            return None
        except ValueError:
            print("[ERROR] Failed to decode JSON.")
            return None

In [5]:
queries = {
    "health": "health OR medicine OR hospital OR disease OR mental health OR vaccine OR healthcare",
    "entertainment": "movie OR tv OR music OR celebrity OR film OR entertainment OR showbiz OR pop culture",
    "politics": "election OR government OR policy OR president OR congress OR parliament OR democracy",
    "sports": "football OR soccer OR olympics OR sports OR nba OR tennis OR world cup OR athlete",
    "business": "business OR finance OR market OR economy OR inflation OR investment OR stock OR gdp"
}

In [6]:
queries.keys()

dict_keys(['health', 'entertainment', 'politics', 'sports', 'business'])

In [ ]:
for i in queries.keys():
    for page in range(1, 3):
        data = fetch_news(
                    api_url = API_URL,
                    params = {
                        "q": queries[i],
                        "from": "2025-07-01",
                        "to": "2025-07-01",
                        "language": "en",
                        "sortBy": "publishedAt",
                        "pageSize": PAGE_SIZE,
                        "page": page
                    },
                    headers={"X-Api-Key": API_KEY}
                )
        if data:
            print(f'Category: {i}')
            print(data['status'])
            print(data['totalResults'])
            print(len(data['articles']))
            print(data['articles'][:2])
            print('-'*20)

In [ ]:
FCSAPI_URL = "https://news.fcsapi.com/api/news"
FCSAPI_KEY = os.getenv('fcsapi_apikey')

for i in ['politics', 'sports', 'business']:
    first = True
    print(f'Category: {i}')
    for j in range(0, 10000, PAGE_SIZE):
        data, headers_reponse = fetch_news(
                    api_url = FCSAPI_URL,
                    params = {
                        "access_key": FCSAPI_KEY,
                        "category": i,
                        # "from": "2025-07-01",
                        # "to": "2025-07-01",
                        "language": "en",
                        "limit": 100,
                        "offset": j
                    },
                    headers = {}
                )
        if first:
            first = False
            print(data['response'][:2])
            print(headers_reponse)

        if data['status'] == False:
            print(f'Final amount: {previous_amount + j}')
            print('-'*20)
            break
        
        previous_amount = len(data['response'])

In [ ]:
load_dotenv(override=True)
WORLDNEWS_URL = "https://api.worldnewsapi.com/search-news"
WORLDNEWS_KEY = os.getenv('worldnews_apikey')

for i in queries.keys():
    print(f'Category: {i}')
    params = {
                "categories": 'sports',
                "language": "en",
                "number": 100,
                # "earliest-publish-date": "2025-07-01",
                "sort":"publish-time",
                "sort-direction":"DESC"
            }
    
    headers = {'x-api-key': WORLDNEWS_KEY}

    print(params)
    data, headers = fetch_news(
                api_url = WORLDNEWS_URL,
                params = params,
                headers = headers
            )
    
    print(headers['X-API-Quota-Left'])
    print(data)
    break
    
    print(data['available'])
    print(len(data['news']))
    print('-'*20)